In [2]:
import pandas as pd
import numpy as np


In [3]:
dataset = pd.read_csv(r"C:\Rahul KM Ghosh_PGD202564310\Innomatics Research Lab\GEN_AI\ASSIGENEMT_NLP_04\IMDB Dataset.csv")

In [4]:
dataset

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [5]:
print(dataset.columns)

Index(['review', 'sentiment'], dtype='object')


In [6]:
dataset = dataset.rename(columns={"review": "text", "sentiment": "label"})

In [7]:
import re

def clean_text(text):
    text = re.sub(r"<.*?>", "", text)           # remove HTML tags
    text = re.sub(r"[^a-zA-Z\s]", "", text)    # remove non-alphabetic characters
    text = text.lower()                         # lowercase
    return text

dataset["text"] = dataset["text"].apply(clean_text)

# Check
print(dataset.head())

                                                text     label
0  one of the other reviewers has mentioned that ...  positive
1  a wonderful little production the filming tech...  positive
2  i thought this was a wonderful way to spend ti...  positive
3  basically theres a family where a little boy j...  negative
4  petter matteis love in the time of money is a ...  positive


In [8]:
label_mapping = {"positive": 1, "negative": 0}
dataset["label"] = dataset["label"].map(label_mapping)

In [9]:
dataset.sample

<bound method NDFrame.sample of                                                     text  label
0      one of the other reviewers has mentioned that ...      1
1      a wonderful little production the filming tech...      1
2      i thought this was a wonderful way to spend ti...      1
3      basically theres a family where a little boy j...      0
4      petter matteis love in the time of money is a ...      1
...                                                  ...    ...
49995  i thought this movie did a down right good job...      1
49996  bad plot bad dialogue bad acting idiotic direc...      0
49997  i am a catholic taught in parochial elementary...      0
49998  im going to have to disagree with the previous...      0
49999  no one expects the star trek movies to be high...      0

[50000 rows x 2 columns]>

In [10]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    dataset["text"], dataset["label"], test_size=0.1, random_state=42
)

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=128)

c:\Users\Tech Lab 1\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels.iloc[idx])
        return item

train_dataset = TextDataset(train_encodings, train_labels)
val_dataset = TextDataset(val_encodings, val_labels)

In [18]:
pip install --upgrade transformers

Note: you may need to restart the kernel to use updated packages.


In [14]:
evaluation_strategy="epoch"

In [15]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6899.38it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tr

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [21]:
import transformers
print(transformers.__version__)

5.5.0


In [22]:
import sys
print(sys.executable)

c:\Users\Tech Lab 1\AppData\Local\Programs\Python\Python312\python.exe


In [23]:
import transformers
print(transformers.__version__)


5.5.0


In [24]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.model_selection import train_test_split

# 1. Load your dataset (already in pandas)
# dataset = pd.read_csv("your_dataset.csv")

# 2. Convert sentiment labels to integers
dataset['label'] = dataset['sentiment'].map({'positive': 1, 'negative': 0})

# 3. Hugging Face Dataset
dataset_hf = Dataset.from_pandas(dataset)

# 4. Train-test split
train_test = dataset_hf.train_test_split(test_size=0.2)
train_dataset = train_test['train']
eval_dataset = train_test['test']

# 5. Tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch['review'], padding=True, truncation=True)

train_dataset = train_dataset.map(tokenize, batched=True)
eval_dataset = eval_dataset.map(tokenize, batched=True)

# 6. Load BERT model
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# 7. TrainingArguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
)

# 8. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

# 9. Train
trainer.train()

# 10. Evaluate
metrics = trainer.evaluate()
print(metrics)

ModuleNotFoundError: No module named 'datasets'

In [25]:
import transformers
import datasets
import accelerate

print(transformers.__version__)  # Should be 5.5.0
print(datasets.__version__)      # Should show installed version
print(accelerate.__version__)    # Should show installed version

ModuleNotFoundError: No module named 'datasets'

In [20]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",  # modern syntax works now
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",         # optional for tensorboard
)

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'